# Loan Limit Increase Optimization

A 60-day **Markov Decision Process (MDP)** for optimizing loan limit increase offers under a regulatory capital constraint, calibrated on `data/loan_limit_increases.csv` (30,000 customer records) and validated via backward/forward dynamic programming equivalence and a 1,000-replication Monte Carlo simulation.

### Assessment Rules & Model Scope
- **Horizon & Epoch**: $K = 6$ decision epochs per year (one epoch $= 60$ days; eligibility requires $\ge 60$ days since last loan).
- **Unit Economics**: $\$40$ profit contribution per accepted increase; $19\%$ annual discount rate converted once to a 60-day discount factor.
- **Capital Constraint**: Expected new exposure capped at $30\%$ of the initial loan book (`capital_frac = 0.30`).
- **Risk & Macro Structure**: Three risk tiers (`Prime`, `Near-Prime`, `Sub-Prime`) with Markov tier migration, plus three macro regimes (`base`, `benign`, `stress`).

### Notebook Roadmap
1. **Model Functions, Assumptions & Dynamics:** formal mathematical formulation, assumption rationales, calibration, single-regime MDP, Monte Carlo simulator, and stochastic macro-regime MDP.
2. **Data Audit & Calibration:** historical rule verification, uptake signal check, and tier risk calibration.
3. **Policy Optimization Across Macro Scenarios:** benchmark heuristics vs. unconstrained and capital-constrained DP policies.
4. **Validation:** backward/forward DP equivalence ($\lambda = 0$), 1,000-replication Monte Carlo check, and macro-MDP shadow-price calibration.
5. **Sensitivity Analysis & Innovative Strategies:** credit-assumption grid (`late_to_default` $\times$ `lgd`), borrower-behavior ablation (churn, early repayment, acceptance), and risk-tiered variable step sizing.
6. **Efficient Frontier & Visual Diagnostics:** modernized `ggplot2 >= 4.0` charts and combined `patchwork` dashboard.
7. **Headline Summary & Operationalization:** core comparison and production deployment recommendations.


In [1]:
# Core packages: tidyverse + yardstick for calibration, knitr for tables,
# scales/patchwork/ggrepel for ggplot2 4.0+ visualizations, matrixStats for MC draws.
librarian::shelf(
  tidyverse, yardstick, knitr, scales,
  patchwork, ggrepel, matrixStats
)
options(ark.plot.width = 10, ark.plot.height = 5)


## 1. Model Functions, Assumptions & Dynamics

The assessment dataset (`data/loan_limit_increases.csv`) has 6 columns: `Customer ID`, `Initial Loan ($)`, `Days Since Last Loan`, `On-time Payments (%)`, `No. of Increases in 2023`, and `Total Profit Contribution ($)`. It does not record default outcomes, loss given default (LGD), limit step size, early repayment, churn, tier transitions, or macroeconomic elasticities. We therefore:
- Calibrate **tier mix** and **per-offer acceptance** directly from the dataset.
- Enforce the **assessment rules** ($\$40$/increase, $19\%$ annual discount, 60-day cycle, max 6/year, 30% capital cap).
- State all unobserved parameters explicitly in `default_assumptions` and `scenario_table`, and stress-test them in Section 5.


### 1.1 Mathematical Formulation, Parameters & Assumption Rationales

#### Formal MDP Formulation
For each customer $i \in \{1, \dots, N\}$ and decision epoch $k \in \{1, \dots, K\}$ ($K = 6$, each epoch $= 60$ days, discount factor $\beta = (1 + 0.19)^{-60/365} \approx 0.97181$):
- **State**: $s_{i,k} \in \{1\text{ (Prime)}, 2\text{ (Near-Prime)}, 3\text{ (Sub-Prime)}\}$, plus absorbing states $\text{Default}$ and $\text{Churn}$.
- **Action**: $a_{i,k} \in \{0\text{ (Hold)}, 1\text{ (Offer)}\}$, restricted to $a_{i,k} = 0$ when ineligible ($e_{i,k} = 0$).
- **Immediate Expected Reward & Exposure**: For state-contingent incremental exposure $\text{EAD}_{i,s} = \text{increase\_frac}_s \cdot L_{i,0}$, unit profit $\pi_s = 40 \cdot (\text{increase\_frac}_s / 0.25)$, and acceptance probability $p_{\text{acc}}$:
  $$R_i(s, 0) = 0, \qquad R_i(s, 1) = p_{\text{acc}} \left(\pi_s - \beta \cdot \text{LGD} \cdot \text{EAD}_{i,s} \cdot \text{PD}_s\right), \qquad E_i(s, 1) = p_{\text{acc}} \cdot \text{EAD}_{i,s}$$
- **Bellman Optimality & Lagrangian Dual**: Under undiscounted regulatory capital cap $C_{\text{cap}} = 0.30 \sum_i L_{i,0}$ and shadow price $\lambda \ge 0$ (scaled in current-value epoch-$k$ units as $\lambda_k = \lambda \beta^{-(k-1)}$):
  $$V_k^{\lambda}(s) = \max_{a \in \mathcal{A}_{i,k}} \left\{ R_i(s, a) - \lambda_k E_i(s, a) + \beta \sum_{s'} P(s' \mid s, a) V_{k+1}^{\lambda}(s') \right\}, \qquad V_{K+1}^{\lambda}(s) = 0$$
  where $P(s' \mid s, 0) = (1 - c_{\text{hold}}) P_{\text{hold}}(s, s')$ and $P(s' \mid s, 1)$ blends the declined branch $(1 - p_{\text{acc}})(1 - c_{\text{dec}}) P_{\text{hold}}$ with the surviving accepted branch $p_{\text{acc}}(1 - \text{PD}_s)(1 - c_{\text{acc}}) M_{\text{repay}}(s, s')$. Conditional on survival ($1 - \text{PD}_s$), $M_{\text{repay}}$ weights non-defaulting late repayments by $w_{\text{late}, s} = \frac{(1 - \text{late\_to\_default})(1 - \overline{\text{OTP}}_s/100)}{1 - \text{PD}_s}$ into downgrade matrix $P_{\text{late}}$, and routes the remaining $1 - w_{\text{late}, s}$ into $p_{\text{early}} P_{\text{early}} + (1 - p_{\text{early}}) P_{\text{ontime}}$.

#### Economic Rationale & Modeling Scope Caveats
- **Churn ordering (`churn_accepted = 0.01 < churn_declined = 0.03 < churn_hold = 0.04`)**: Customers who receive and accept a limit increase have stronger relationship engagement; even receiving an offer (`declined`) signals proactive service relative to receiving no offer (`hold`).
- **Tier migration (`p_early_mat`, `p_ontime_mat`, `p_hold`)**: Early repayment upgrades Near-Prime $\to$ Prime and Sub-Prime $\to$ Near-Prime at $35\%$ per cycle; on-time repayment has symmetric $5\%$–$10\%$ drift; holding preserves tier with $94\%$–$97\%$ persistence.
- **Scope limitations**: (1) *Variable term extension* is omitted (fixed 60-day cycle) because term-extension days are unobserved in the CSV; (2) *Eligibility* enforces `days_since_last_loan >= 60` at $t=0$ and assumes 60-day spacing thereafter (relying on the MDP rather than a hard tier gate); (3) *Incremental exposure* evaluates default/LGD on the new limit slice only and accumulates gross new exposure; (4) *Terminal value* $V_7 = 0$ is a 1-year boundary mitigated in production via rolling-horizon re-solving.


In [2]:
tier_labels <- c("Prime", "Near-Prime", "Sub-Prime")

default_assumptions <- list(
  # Assessment-specified rules
  profit_per_increase = 40,
  annual_discount = 0.19,
  cycle_days = 60,
  n_epochs = 6L,
  capital_frac = 0.30,    # cap on expected new exposure as share of initial loan book

  # Explicit modelling assumptions (unobserved in dataset)
  increase_frac = 0.25,   # each accepted increase raises limit by 25% of initial loan
  lgd = 0.55,             # unsecured consumer lending LGD benchmark
  late_to_default = 0.6,  # P(default in cycle) = late_to_default * tier late-payment rate
  p_early = 0.20,         # P(repay early | no default)
  churn_hold = 0.04,      # per-cycle churn when no offer is made
  churn_declined = 0.03,  # per-cycle churn when offer is declined
  churn_accepted = 0.01   # improved retention when increase is accepted
)

# Macro scenarios: percentage-point deviations from base case.
# Sensitivities are assumed elasticities for acceptance logit shift and PD multiplier.
scenario_table <- tribble(
  ~scenario, ~inflation_pp, ~unemployment_pp, ~rate_pp,
  "base",    0,             0,                0,
  "benign", -1,            -1,               -1,
  "stress",  3,             2,                2
) |>
  mutate(
    accept_logit_shift = -0.05 * inflation_pp - 0.10 * unemployment_pp -
      0.08 * rate_pp,
    pd_mult = exp(0.03 * inflation_pp + 0.08 * unemployment_pp +
      0.05 * rate_pp),
    scenario = fct_inorder(scenario)
  )

# Tier migration matrices: rows = from tier, columns = to tier
p_hold <- matrix(
  c(
    0.97, 0.03, 0.00,
    0.03, 0.94, 0.03,
    0.00, 0.05, 0.95
  ),
  nrow = 3, byrow = TRUE
)

p_early_mat <- matrix(
  c(
    1.00, 0.00, 0.00,
    0.35, 0.65, 0.00,
    0.00, 0.35, 0.65
  ),
  nrow = 3, byrow = TRUE
)

p_ontime_mat <- matrix(
  c(
    0.95, 0.05, 0.00,
    0.05, 0.90, 0.05,
    0.00, 0.10, 0.90
  ),
  nrow = 3, byrow = TRUE
)

p_late_mat <- matrix(
  c(
    0.65, 0.35, 0.00,
    0.00, 0.65, 0.35,
    0.00, 0.00, 1.00
  ),
  nrow = 3, byrow = TRUE
)

# Macro-regime Markov transition matrix: rows = current regime (base, benign, stress),
# columns = next 60-day regime.
p_macro <- matrix(
  c(
    0.85, 0.08, 0.07,
    0.15, 0.80, 0.05,
    0.15, 0.05, 0.80
  ),
  nrow = 3, byrow = TRUE
)


### 1.2 Data Loading & Parameter Calibration
Functions to load the 30,000-row dataset, assign initial risk tiers from `On-time Payments (%)`, calibrate per-offer acceptance and tier PDs, and assemble the `world` parameter object.


In [3]:
#' Load and standardize the customer dataset with initial risk tiers
load_customers <- function(path) {
  read_csv(path, show_col_types = FALSE) |>
    rename(
      customer_id = `Customer ID`,
      initial_loan = `Initial Loan ($)`,
      days_since_last_loan = `Days Since Last Loan`,
      on_time_pct = `On-time Payments (%)`,
      n_increases = `No. of Increases in 2023`,
      profit_hist = `Total Profit Contribution ($)`
    ) |>
    mutate(
      tier = case_when(
        on_time_pct >= 95 ~ 1L,
        on_time_pct >= 85 ~ 2L,
        .default = 3L
      )
    )
}

#' Estimate per-offer acceptance probability from 2023 historical uptake
estimate_acceptance <- function(customers, n_epochs, adjust_attrition = FALSE) {
  elig_ep1 <- mean(pull(customers, days_since_last_loan) >= 60)
  eff_epochs <- if (adjust_attrition) (elig_ep1 + (n_epochs - 1L)) * 0.8828 else n_epochs
  customers |>
    summarise(
      share_active = mean(n_increases > 0),
      mean_when_active = mean(n_increases[n_increases > 0])
    ) |>
    mutate(accept_per_offer = share_active * mean_when_active / eff_epochs)
}

#' Estimate 60-day cycle default and non-default delinquency probability by risk tier
estimate_tier_pd <- function(customers, late_to_default) {
  customers |>
    summarise(mean_otp = mean(on_time_pct), n = n(), .by = tier) |>
    arrange(tier) |>
    mutate(
      share = n / sum(n),
      late_cycle = 1 - mean_otp / 100,
      pd_cycle = late_to_default * late_cycle,
      delinq_cycle = (1 - late_to_default) * late_cycle
    )
}

#' Build a complete 60-day MDP world object for a given macro scenario
build_world <- function(customers, p, accept_logit_shift = 0, pd_mult = 1) {
  n <- nrow(customers)
  accept_base <- estimate_acceptance(
    customers, p$n_epochs, isTRUE(p$adjust_attrition)
  ) |>
    pull(accept_per_offer)
  tier_tbl <- estimate_tier_pd(customers, p$late_to_default)
  tier_pd <- pull(tier_tbl, pd_cycle)
  tier_delinq <- pull(tier_tbl, delinq_cycle)

  pd_scen <- pmin(tier_pd * pd_mult, 0.6)
  delinq_scen <- pmin(tier_delinq * pd_mult, 0.95 - pd_scen)

  list(
    p = p,
    n = n,
    k = p$n_epochs,
    tier0 = pull(customers, tier),
    ead = outer(pull(customers, initial_loan), rep(p$increase_frac, length.out = 3L)),
    # Only customers with >= 60 days since last loan are eligible at epoch 1
    elig = cbind(
      pull(customers, days_since_last_loan) >= p$cycle_days,
      matrix(TRUE, n, p$n_epochs - 1L)
    ),
    accept = plogis(qlogis(accept_base) + accept_logit_shift),
    pd = pd_scen,
    w_late = delinq_scen / (1 - pd_scen),
    disc = (1 + p$annual_discount)^(-p$cycle_days / 365),
    cap = p$capital_frac * sum(pull(customers, initial_loan))
  )
}

#' Blended repayment tier-transition matrix (early + on-time + non-default late)
repay_matrix <- function(world) {
  m_good <- world$p$p_early * p_early_mat + (1 - world$p$p_early) * p_ontime_mat
  w_late <- if (!is.null(world$w_late) && isTRUE(world$p$use_late_downgrade)) {
    world$w_late
  } else {
    rep(0, 3L)
  }
  sweep(m_good, 1, 1 - w_late, `*`) + sweep(p_late_mat, 1, w_late, `*`)
}


### 1.3 Single-Regime MDP Solver & Analytic Evaluator
Backward dynamic programming solver (`solve_policy`) using Lagrangian relaxation (`lambda` = shadow price on expected new exposure; `lambda = 0` is unconstrained), paired with exact forward state-distribution evaluation (`evaluate_policy`), root-finding for the binding capital shadow price (`solve_constrained`), and static tier-rule baselines (`static_policy`).


In [4]:
#' Solve finite-horizon MDP via backward induction for a given shadow price lambda
solve_policy <- function(world, lambda = 0) {
  p <- world$p
  m_repay <- repay_matrix(world)
  loss_term <- p$lgd * world$disc * sweep(world$ead, 2, world$pd, `*`)

  prof_mat <- matrix(rep(p$profit_per_increase, length.out = 3L), world$n, 3L, byrow = TRUE)

  step <- function(state, kk) {
    v_next <- state$v
    cont_hold <- v_next %*% t(p_hold)
    q_hold <- world$disc * (1 - p$churn_hold) * cont_hold
    q_declined <- world$disc * (1 - p$churn_declined) * cont_hold
    cont_acc <- (v_next %*% t(m_repay)) %*% diag(1 - world$pd)
    q_acc <- prof_mat - loss_term +
      world$disc * (1 - p$churn_accepted) * cont_acc
    lambda_k <- lambda / (world$disc^(kk - 1L))
    q_offer <- world$accept * q_acc + (1 - world$accept) * q_declined -
      lambda_k * world$accept * world$ead
    offer <- (q_offer > q_hold) & world$elig[, kk]
    list(
      v = q_hold + offer * (q_offer - q_hold),
      offers = c(list(offer), state$offers)
    )
  }

  out <- reduce(
    rev(seq_len(world$k)),
    step,
    .init = list(v = matrix(0, world$n, 3), offers = list())
  )
  list(
    offers = out$offers,
    v1 = out$v[cbind(seq_len(world$n), world$tier0)]
  )
}

#' Propagate exact tier probability masses forward under a given offer policy
evaluate_policy <- function(offers, world) {
  p <- world$p
  m_repay <- repay_matrix(world)
  dist0 <- matrix(0, world$n, 3)
  dist0[cbind(seq_len(world$n), world$tier0)] <- 1

  prof_vec <- rep(p$profit_per_increase, length.out = 3L)

  step <- function(st, kk) {
    mass_offered <- st$dist * (offers[[kk]] & world$elig[, kk])
    accepted <- world$accept * mass_offered
    declined <- (1 - world$accept) * mass_offered
    idle <- st$dist - mass_offered

    survivors <- (accepted %*% diag(1 - world$pd)) * (1 - p$churn_accepted)
    new_dist <- (idle * (1 - p$churn_hold) +
      declined * (1 - p$churn_declined)) %*% p_hold +
      survivors %*% m_repay

    n_accepted <- rowSums(accepted)
    p_default <- rowSums(accepted %*% diag(world$pd))
    prof_accepted <- as.vector(accepted %*% prof_vec)
    ead_accepted <- rowSums(accepted * world$ead)
    ead_default <- rowSums((accepted * world$ead) %*% diag(world$pd))
    time_disc <- world$disc^(kk - 1L)

    list(
      dist = new_dist,
      profit = st$profit + time_disc * prof_accepted,
      loss = st$loss + time_disc * world$disc * p$lgd * ead_default,
      exposure = st$exposure + ead_accepted,
      n_accepted = st$n_accepted + n_accepted,
      p_default = st$p_default + p_default,
      n_offers = st$n_offers + rowSums(mass_offered)
    )
  }

  zero <- rep(0, world$n)
  reduce(
    seq_len(world$k),
    step,
    .init = list(
      dist = dist0, profit = zero, loss = zero, exposure = zero,
      n_accepted = zero, p_default = zero, n_offers = zero
    )
  )
}

#' Aggregate customer-level forward evaluation into portfolio KPIs
summarise_evaluation <- function(ev, world) {
  tibble(
    npv = sum(ev$profit - ev$loss),
    gross_profit = sum(ev$profit),
    credit_loss = sum(ev$loss),
    exposure = sum(ev$exposure),
    exposure_vs_cap = sum(ev$exposure) / world$cap,
    accepted_increases = sum(ev$n_accepted),
    expected_defaults = sum(ev$p_default),
    default_rate_per_accepted = sum(ev$p_default) / sum(ev$n_accepted),
    customers_ever_offered = sum(ev$n_offers > 0)
  )
}

#' Find the smallest non-negative shadow price lambda* satisfying the capital cap
solve_constrained <- function(world) {
  gap_at <- \(lambda) {
    sum(evaluate_policy(solve_policy(world, lambda)$offers, world)$exposure) - world$cap
  }
  if (gap_at(0) <= 0) {
    return(0)
  }
  find_upper <- \(h, iter = 1L) {
    if (gap_at(h) <= 0 || iter >= 40L) h else find_upper(h * 2, iter + 1L)
  }
  upper <- find_upper(0.05)
  root <- stats::uniroot(gap_at, interval = c(0, upper), tol = 1e-8)$root
  if (gap_at(root) > 0) root + 1e-6 else root
}

#' Construct a time-invariant tier-based heuristic offer policy
static_policy <- function(world, allowed_tiers) {
  allowed <- matrix(
    tier_labels %in% allowed_tiers,
    nrow = world$n, ncol = 3, byrow = TRUE
  )
  rep(list(allowed), world$k)
}


### 1.4 Monte Carlo Simulator
Customer-level stochastic simulator (`simulate_policy`) using the exact same 60-day probabilities, discount factor, and transition matrices to verify the analytic DP solution and measure tail risk.


In [5]:
#' Sample next risk tier from cumulative transition probabilities
draw_next_tier <- function(state, cum) {
  u <- runif(length(state))
  1L + (u > cum[cbind(state, 1L)]) + (u > cum[cbind(state, 2L)])
}

#' Execute one 60-day Monte Carlo step across all customers
mc_step <- function(st, kk, offers, world, cums) {
  p <- world$p
  n <- world$n
  prof_vec <- rep(p$profit_per_increase, length.out = 3L)
  idx <- cbind(seq_len(n), st$state)
  offered <- st$alive & world$elig[, kk] & offers[[kk]][idx]
  accepted <- offered & (runif(n) < world$accept)
  declined <- offered & !accepted
  defaulted <- accepted & (runif(n) < world$pd[st$state])
  churn_p <- case_when(
    accepted ~ p$churn_accepted,
    declined ~ p$churn_declined,
    .default = p$churn_hold
  )
  churned <- st$alive & !defaulted & (runif(n) < churn_p)
  next_state <- if_else(
    accepted,
    draw_next_tier(st$state, cums$repay),
    draw_next_tier(st$state, cums$hold)
  )
  time_disc <- world$disc^(kk - 1L)
  list(
    state = next_state,
    alive = st$alive & !defaulted & !churned,
    npv = st$npv + time_disc * (
      sum(prof_vec[st$state[accepted]]) -
        world$disc * p$lgd * sum(world$ead[idx][defaulted])
    )
  )
}

#' Run n_reps seeded Monte Carlo portfolio replications
simulate_policy <- function(offers, world, n_reps = 200L, seed = 2023L) {
  set.seed(seed)
  cums <- list(
    hold = matrixStats::rowCumsums(p_hold),
    repay = matrixStats::rowCumsums(repay_matrix(world))
  )
  one_rep <- \(rep_id) {
    init <- list(state = world$tier0, alive = rep(TRUE, world$n), npv = 0)
    reduce(
      seq_len(world$k),
      \(st, kk) mc_step(st, kk, offers, world, cums),
      .init = init
    )$npv
  }
  map_dbl(seq_len(n_reps), one_rep)
}


### 1.5 Stochastic Macro-Regime MDP
Jointly optimizes over `(risk tier, macro regime)` with regime transitions governed by `p_macro`.


In [6]:
#' Single-regime Bellman backup given expected continuation values ev_next
macro_dp_step_regime <- function(m, kk, ev_next, worlds, m_repays, loss_terms, lambda) {
  w <- worlds[[m]]
  p <- w$p
  vn <- ev_next[[m]]
  prof_mat <- matrix(rep(p$profit_per_increase, length.out = 3L), w$n, 3L, byrow = TRUE)
  cont_hold <- vn %*% t(p_hold)
  q_hold <- w$disc * (1 - p$churn_hold) * cont_hold
  q_declined <- w$disc * (1 - p$churn_declined) * cont_hold
  cont_acc <- (vn %*% t(m_repays[[m]])) %*% diag(1 - w$pd)
  q_acc <- prof_mat - loss_terms[[m]] +
    w$disc * (1 - p$churn_accepted) * cont_acc
  lambda_k <- lambda / (w$disc^(kk - 1L))
  q_offer <- w$accept * q_acc + (1 - w$accept) * q_declined -
    lambda_k * w$accept * w$ead
  offer <- (q_offer > q_hold) & w$elig[, kk]
  list(v = q_hold + offer * (q_offer - q_hold), offer = offer)
}

#' Solve the joint (tier x macro-regime) finite-horizon MDP
solve_macro_mdp <- function(customers, p, scen_tbl = scenario_table,
                            macro_mat = p_macro, lambda = 0, worlds = NULL) {
  if (is.null(worlds)) {
    worlds <- scen_tbl |>
      mutate(w = map2(
        accept_logit_shift, pd_mult,
        \(s, m) build_world(customers, p, s, m)
      )) |>
      pull(w)
  }
  w1 <- worlds[[1]]
  m_repays <- map(worlds, repay_matrix)
  loss_terms <- map(worlds, \(w) p$lgd * w$disc * sweep(w$ead, 2, w$pd, `*`))

  step <- function(state, kk) {
    v_next <- state$v
    ev_next <- map(seq_len(3L), \(m) {
      macro_mat[m, 1] * v_next[[1]] +
        macro_mat[m, 2] * v_next[[2]] +
        macro_mat[m, 3] * v_next[[3]]
    })
    out_m <- map(seq_len(3L), \(m) {
      macro_dp_step_regime(m, kk, ev_next, worlds, m_repays, loss_terms, lambda)
    })
    list(v = map(out_m, "v"), offers = c(list(map(out_m, "offer")), state$offers))
  }

  out <- reduce(
    rev(seq_len(w1$k)),
    step,
    .init = list(v = rep(list(matrix(0, w1$n, 3)), 3L), offers = list())
  )
  list(
    offers = out$offers,
    v1_base = out$v[[1]][cbind(seq_len(w1$n), w1$tier0)],
    worlds = worlds
  )
}

#' Single forward propagation step for the stochastic macro-regime MDP
macro_eval_step <- function(st, kk, macro_sol, macro_mat, m_repays) {
  worlds <- macro_sol$worlds
  w1 <- worlds[[1]]
  p <- w1$p
  prof_vec <- rep(p$profit_per_increase, length.out = 3L)
  per_m <- map(seq_len(3L), \(m) {
    w <- worlds[[m]]
    d <- st$dists[[m]]
    mass_off <- d * (macro_sol$offers[[kk]][[m]] & w$elig[, kk])
    acc <- w$accept * mass_off
    dec <- (1 - w$accept) * mass_off
    idle <- d - mass_off
    surv <- (acc %*% diag(1 - w$pd)) * (1 - p$churn_accepted)
    next_d <- (idle * (1 - p$churn_hold) + dec * (1 - p$churn_declined)) %*%
      p_hold + surv %*% m_repays[[m]]
    list(
      next_d = next_d,
      n_acc = rowSums(acc),
      prof_acc = as.vector(acc %*% prof_vec),
      p_def = rowSums(acc %*% diag(w$pd)),
      ead_acc = rowSums(acc * w$ead),
      ead_def = rowSums((acc * w$ead) %*% diag(w$pd))
    )
  })
  new_dists <- map(seq_len(3L), \(m_to) {
    macro_mat[1, m_to] * per_m[[1]]$next_d +
      macro_mat[2, m_to] * per_m[[2]]$next_d +
      macro_mat[3, m_to] * per_m[[3]]$next_d
  })
  n_acc_tot <- reduce(map(per_m, "n_acc"), `+`)
  prof_acc_tot <- reduce(map(per_m, "prof_acc"), `+`)
  p_def_tot <- reduce(map(per_m, "p_def"), `+`)
  ead_acc_tot <- reduce(map(per_m, "ead_acc"), `+`)
  ead_def_tot <- reduce(map(per_m, "ead_def"), `+`)
  time_disc <- w1$disc^(kk - 1L)
  list(
    dists = new_dists,
    profit = st$profit + time_disc * prof_acc_tot,
    loss = st$loss + time_disc * w1$disc * p$lgd * ead_def_tot,
    exposure = st$exposure + ead_acc_tot,
    n_accepted = st$n_accepted + n_acc_tot,
    p_default = st$p_default + p_def_tot
  )
}

#' Forward evaluation of the stochastic macro-regime MDP from a starting regime
evaluate_macro_mdp <- function(macro_sol, macro_mat = p_macro, start_regime = 1L) {
  w1 <- macro_sol$worlds[[1]]
  m_repays <- map(macro_sol$worlds, repay_matrix)
  init_dists <- rep(list(matrix(0, w1$n, 3)), 3L)
  init_dists[[start_regime]][cbind(seq_len(w1$n), w1$tier0)] <- 1
  zero <- rep(0, w1$n)
  init <- list(
    dists = init_dists, profit = zero, loss = zero,
    exposure = zero, n_accepted = zero, p_default = zero
  )
  ev <- reduce(
    seq_len(w1$k),
    \(st, kk) macro_eval_step(st, kk, macro_sol, macro_mat, m_repays),
    .init = init
  )
  tibble(
    npv = sum(ev$profit - ev$loss),
    gross_profit = sum(ev$profit),
    credit_loss = sum(ev$loss),
    exposure_vs_cap = sum(ev$exposure) / w1$cap,
    accepted_increases = sum(ev$n_accepted),
    default_rate_per_accepted = sum(ev$p_default) / sum(ev$n_accepted)
  )
}


## 2. Data Audit & Calibration

We load the 30,000-customer portfolio, verify the historical profit formula (`Total Profit = 40 * (increases - 2)`) and historical eligibility violations (`days_since_last_loan < 60`), test whether uptake correlates with the three observed covariates, and calibrate per-offer acceptance and tier default probabilities.

- **Historical `- 2` offset**: In the 2023 log, every active account satisfies `profit_hist = 40 * (n_increases - 2)`, implying an $80 historical acquisition/origination hurdle. Forward-looking optimization follows the assessment rule of $40 gross profit per accepted increase.
- **Uptake distribution**: Roughly 44% of customers had 0 increases while 56% averaged ~4.0 increases (`accept_per_offer = 0.3729`). We calibrate the homogeneous per-offer rate to match total 2023 volume and discuss two-component Bayesian responder updating in Section 7.


In [7]:
dir.create("outputs", showWarnings = FALSE)
dir.create("figures", showWarnings = FALSE)

customers <- load_customers("data/loan_limit_increases.csv")
p <- default_assumptions

audit <- tibble(
  rows = nrow(customers),
  mean_loan = mean(pull(customers, initial_loan)),
  share_with_increases = mean(pull(customers, n_increases) > 0),
  historical_profit_total = sum(pull(customers, profit_hist)),
  profit_rule_holds = customers |>
    filter(n_increases > 0) |>
    summarise(ok = all(profit_hist == 40 * (n_increases - 2))) |>
    pull(ok),
  historical_eligibility_violations = customers |>
    filter(days_since_last_loan < 60, n_increases > 0) |>
    nrow()
)

print(kable(audit, digits = 3, format.args = list(big.mark = ",")))

uptake_fit <- glm(
  I(n_increases > 0) ~ on_time_pct + initial_loan + days_since_last_loan,
  data = customers,
  family = binomial()
)

uptake_auc <- customers |>
  mutate(
    pred = predict(uptake_fit, type = "response"),
    y_fct = factor(n_increases > 0, levels = c(TRUE, FALSE))
  ) |>
  yardstick::roc_auc(truth = y_fct, pred, event_level = "first") |>
  pull(.estimate)

cat(sprintf(
  "Uptake logistic AUC from the 3 covariates: %.3f (0.5 = no signal)\n",
  uptake_auc
))

acceptance <- estimate_acceptance(customers, p$n_epochs)
tier_table <- estimate_tier_pd(customers, p$late_to_default)

cat("Acceptance calibration:\n")
print(kable(acceptance, digits = 4))

cat("\nTiers (data-derived mix, assumption-derived PD per 60-day cycle):\n")
kable(tier_table, digits = 4, format.args = list(big.mark = ","))




|   rows| mean_loan| share_with_increases| historical_profit_total|profit_rule_holds | historical_eligibility_violations|
|------:|---------:|--------------------:|-----------------------:|:-----------------|---------------------------------:|
| 30,000| 2,752.533|                 0.56|               1,341,560|TRUE              |                             2,726|


Uptake logistic AUC from the 3 covariates: 0.505 (0.5 = no signal)
Acceptance calibration:


| share_active| mean_when_active| accept_per_offer|
|------------:|----------------:|----------------:|
|       0.5598|           3.9972|           0.3729|

Tiers (data-derived mix, assumption-derived PD per 60-day cycle):




| tier| mean_otp|      n|  share| late_cycle| pd_cycle| delinq_cycle|
|----:|--------:|------:|------:|----------:|--------:|------------:|
|    1|  97.4497|  7,520| 0.2507|     0.0255|   0.0153|       0.0102|
|    2|  90.0063| 15,103| 0.5034|     0.0999|   0.0600|       0.0400|
|    3|  82.4916|  7,377| 0.2459|     0.1751|   0.1051|       0.0700|

## 3. Solving Constrained & Benchmark Policies Across Macro Scenarios

We solve for the binding shadow price of capital (`lambda*`) on the base scenario, construct benchmark policies (`no_increase`, `offer_all`, `prime_only`, `prime_near`, `dp_unconstrained`, `dp_constrained`), and evaluate both fixed-base policies and re-solved adaptive policies across `base`, `benign`, and `stress` scenarios.


In [8]:
base_row <- filter(scenario_table, scenario == "base")
world_base <- build_world(customers, p)

cat(sprintf(
  "Discount factor per 60 days: %.5f | Capital cap: $%s\n",
  world_base$disc, comma(world_base$cap)
))

lambda_star <- solve_constrained(world_base)
cat(sprintf("Shadow price of capital (lambda*): %.4f\n", lambda_star))

policies_base <- list(
  no_increase = static_policy(world_base, character(0)),
  offer_all = static_policy(world_base, tier_labels),
  prime_only = static_policy(world_base, "Prime"),
  prime_near = static_policy(world_base, c("Prime", "Near-Prime")),
  dp_unconstrained = solve_policy(world_base, 0)$offers,
  dp_constrained = solve_policy(world_base, lambda_star)$offers
)

worlds <- scenario_table |>
  mutate(world = map2(
    accept_logit_shift, pd_mult,
    \(s, m) build_world(customers, p, s, m)
  ))

static_names <- c(
  "no_increase", "offer_all", "prime_only",
  "prime_near", "dp_unconstrained", "dp_constrained"
)

results_fixed <- worlds |>
  select(scenario, world) |>
  expand_grid(strategy = static_names) |>
  mutate(res = map2(world, strategy, \(w, s) {
    summarise_evaluation(evaluate_policy(policies_base[[s]], w), w)
  })) |>
  select(-world) |>
  unnest(res) |>
  mutate(policy_basis = "solved on base scenario")

results_adaptive <- worlds |>
  select(scenario, world) |>
  mutate(
    lambda = map_dbl(world, solve_constrained),
    res = map2(world, lambda, \(w, l) {
      summarise_evaluation(evaluate_policy(solve_policy(w, l)$offers, w), w)
    })
  ) |>
  select(-world) |>
  unnest(res) |>
  mutate(
    strategy = "dp_constrained_resolved",
    policy_basis = "re-solved per scenario"
  )

results <- bind_rows(results_fixed, results_adaptive) |>
  mutate(within_cap = exposure_vs_cap <= 1.0001)

write_csv(results, "outputs/strategy_results.csv")

cat("\nBase scenario:\n")
results |>
  filter(scenario == "base") |>
  select(
    strategy, npv, gross_profit, credit_loss, exposure_vs_cap,
    accepted_increases, default_rate_per_accepted
  ) |>
  mutate(
    across(c(npv, gross_profit, credit_loss), round),
    across(c(exposure_vs_cap, default_rate_per_accepted), \(x) round(x, 3))
  ) |>
  kable(digits = 3, format.args = list(big.mark = ",")) |>
  print()

cat("\nOptimal constrained NPV across scenarios (re-solved per scenario):\n")
results |>
  filter(strategy == "dp_constrained_resolved") |>
  select(scenario, npv, accepted_increases, default_rate_per_accepted) |>
  mutate(npv = round(npv)) |>
  kable(digits = 4, format.args = list(big.mark = ","))


Discount factor per 60 days: 0.97181 | Capital cap: $24,772,793


Shadow price of capital (lambda*): 0.0102



Base scenario:


|strategy                |       npv| gross_profit| credit_loss| exposure_vs_cap| accepted_increases| default_rate_per_accepted|
|:-----------------------|---------:|------------:|-----------:|---------------:|------------------:|-------------------------:|
|no_increase             |         0|            0|           0|           0.000|               0.00|                       NaN|
|offer_all               | 1,043,611|    2,179,254|   1,135,643|           1.621|          58,359.51|                     0.057|
|prime_only              |   513,224|      597,721|      84,497|           0.448|          16,052.97|                     0.015|
|prime_near              | 1,011,908|    1,647,345|     635,437|           1.224|          44,123.12|                     0.042|
|dp_unconstrained        | 1,129,713|    1,933,198|     803,485|           1.357|          51,767.07|                     0.051|
|dp_constrained          | 1,084,533|    1,611,601|     527,068|           1.00



|scenario |       npv| accepted_increases| default_rate_per_accepted|
|:--------|---------:|------------------:|-------------------------:|
|base     | 1,084,533|          43,133.11|                    0.0479|
|benign   | 1,275,081|          45,345.31|                    0.0395|
|stress   |   605,370|          28,397.27|                    0.0658|

## 4. Validation

Per the modelling guardrails, results are reportable only if:
1. **Backward DP vs. forward evaluation** agrees at $\lambda = 0$ (both single-regime and stochastic macro-regime MDPs).
2. **Monte Carlo mean** (1,000 seeded replications) sits within $\sim 3$ standard errors of the analytic NPV.


In [9]:
dp_free <- solve_policy(world_base, 0)
fwd_free <- summarise_evaluation(
  evaluate_policy(dp_free$offers, world_base), world_base
)$npv

check_dp <- tibble(
  check = "Backward DP value vs forward evaluation (lambda = 0)",
  analytic_dp = sum(dp_free$v1),
  analytic_forward = fwd_free,
  rel_diff = abs(analytic_dp - analytic_forward) / abs(analytic_dp)
)

dp_constr <- solve_policy(world_base, lambda_star)
fwd_constr <- summarise_evaluation(
  evaluate_policy(dp_constr$offers, world_base), world_base
)

check_dp_constr <- tibble(
  check = "Constrained DP Lagrangian dual vs forward NPV (lambda = lambda*)",
  analytic_dp = sum(dp_constr$v1) + lambda_star * pull(fwd_constr, exposure),
  analytic_forward = pull(fwd_constr, npv),
  rel_diff = abs(analytic_dp - analytic_forward) / abs(analytic_forward)
)

mc_npv <- simulate_policy(policies_base$dp_constrained, world_base, n_reps = 1000L)

analytic_npv <- results |>
  filter(scenario == "base", strategy == "dp_constrained") |>
  pull(npv)

mc_q05 <- as.numeric(quantile(mc_npv, 0.05))
mc_q95 <- as.numeric(quantile(mc_npv, 0.95))

mc_check <- tibble(
  check = "Monte Carlo (1000 reps) vs analytic, dp_constrained, base",
  analytic = analytic_npv,
  mc_mean = mean(mc_npv),
  mc_se = sd(mc_npv) / sqrt(length(mc_npv)),
  z = (mc_mean - analytic) / mc_se,
  mc_p05 = mc_q05,
  mc_cvar_05 = mean(mc_npv[mc_npv <= mc_q05]),
  mc_p95 = mc_q95,
  mc_share_negative = mean(mc_npv < 0)
)

macro_free <- solve_macro_mdp(customers, p, lambda = 0)
macro_free_fwd <- evaluate_macro_mdp(macro_free)

check_macro_dp <- tibble(
  check = "Stochastic macro MDP backward vs forward (lambda = 0)",
  analytic_dp = sum(macro_free$v1_base),
  analytic_forward = pull(macro_free_fwd, npv),
  rel_diff = abs(analytic_dp - analytic_forward) / abs(analytic_dp)
)

macro_constr <- solve_macro_mdp(customers, p, lambda = lambda_star)
macro_constr_eval <- evaluate_macro_mdp(macro_constr)

# Pre-build scenario worlds once and solve for binding lambda_macro*
macro_worlds <- scenario_table |>
  mutate(w = map2(
    accept_logit_shift, pd_mult,
    \(s, m) build_world(customers, p, s, m)
  )) |>
  pull(w)

macro_gap <- \(lam) {
  sol <- solve_macro_mdp(customers, p, lambda = lam, worlds = macro_worlds)
  pull(evaluate_macro_mdp(sol), exposure_vs_cap) - 1
}

lambda_macro_star <- if (macro_gap(0) <= 0) {
  0
} else {
  find_up <- \(h, it = 1L) if (macro_gap(h) <= 0 || it >= 40L) h else find_up(h * 2, it + 1L)
  rt <- stats::uniroot(macro_gap, interval = c(0, find_up(0.05)), tol = 1e-6)$root
  if (macro_gap(rt) > 0) rt + 1e-6 else rt
}
macro_constr_opt_eval <- evaluate_macro_mdp(
  solve_macro_mdp(customers, p, lambda = lambda_macro_star, worlds = macro_worlds)
) |>
  mutate(lambda_macro_star = lambda_macro_star)

bind_rows(
  check_dp |> mutate(across(where(is.numeric), as.numeric)),
  check_dp_constr |> mutate(across(where(is.numeric), as.numeric)),
  check_macro_dp |> mutate(across(where(is.numeric), as.numeric)),
  mc_check |> mutate(across(where(is.numeric), as.numeric))
) |>
  write_csv("outputs/validation.csv")

print(kable(check_dp, digits = 4, format.args = list(big.mark = ",")))
print(kable(check_dp_constr, digits = 4, format.args = list(big.mark = ",")))
print(kable(mc_check, digits = 3, format.args = list(big.mark = ",")))
print(kable(check_macro_dp, digits = 4, format.args = list(big.mark = ",")))

cat("\nStochastic macro-regime MDP (at base lambda*):\n")
print(kable(macro_constr_eval, digits = 4, format.args = list(big.mark = ",")))

cat("\nStochastic macro-regime MDP (re-calibrated at lambda_macro*):\n")
kable(macro_constr_opt_eval, digits = 4, format.args = list(big.mark = ","))




|check                                                | analytic_dp| analytic_forward| rel_diff|
|:----------------------------------------------------|-----------:|----------------:|--------:|
|Backward DP value vs forward evaluation (lambda = 0) |   1,129,713|        1,129,713|        0|


|check                                                            | analytic_dp| analytic_forward| rel_diff|
|:----------------------------------------------------------------|-----------:|----------------:|--------:|
|Constrained DP Lagrangian dual vs forward NPV (lambda = lambda*) |   1,084,533|        1,084,533|        0|


|check                                                     |  analytic|   mc_mean|   mc_se|     z|    mc_p05| mc_cvar_05|    mc_p95| mc_share_negative|
|:---------------------------------------------------------|---------:|---------:|-------:|-----:|---------:|----------:|---------:|-----------------:|
|Monte Carlo (1000 reps) vs analytic, dp_constrained, base | 1,084,533| 



|check                                                 | analytic_dp| analytic_forward| rel_diff|
|:-----------------------------------------------------|-----------:|----------------:|--------:|
|Stochastic macro MDP backward vs forward (lambda = 0) |   1,110,196|        1,110,196|        0|

Stochastic macro-regime MDP (at base lambda*):


|       npv| gross_profit| credit_loss| exposure_vs_cap| accepted_increases| default_rate_per_accepted|
|---------:|------------:|-----------:|---------------:|------------------:|-------------------------:|
| 1,067,634|    1,586,638|   519,003.9|          0.9875|          42,449.82|                     0.048|

Stochastic macro-regime MDP (re-calibrated at lambda_macro*):




|       npv| gross_profit| credit_loss| exposure_vs_cap| accepted_increases| default_rate_per_accepted| lambda_macro_star|
|---------:|------------:|-----------:|---------------:|------------------:|-------------------------:|-----------------:|
| 1,070,721|    1,599,194|   528,472.4|               1|          42,785.93|                    0.0481|            0.0098|

## 5. Sensitivity Analysis & Innovative Strategies

We stress-test:
1. **Credit assumptions** (`late_to_default` $	imes$ `lgd`).
2. **Borrower behavior** (retention lift from `churn_accepted`, early repayment `p_early`, and acceptance shifts).
3. **Innovative strategy: Risk-Tiered Variable Step Sizing**: comparing uniform `25%` limit increases against tier-differentiated steps (`Prime = 35%`, `Near-Prime = 20%`, `Sub-Prime = 10%`) under the same 30% capital cap.


In [10]:
sens <- expand_grid(
  late_to_default = c(0.3, 0.45, 0.6, 0.75, 0.9),
  lgd = c(0.35, 0.55, 0.75)
) |>
  mutate(res = map2(late_to_default, lgd, \(ltd, l) {
    p_i <- list_modify(p, late_to_default = ltd, lgd = l)
    w <- build_world(customers, p_i)
    lam <- solve_constrained(w)
    pol <- solve_policy(w, lam)$offers
    summarise_evaluation(evaluate_policy(pol, w), w) |>
      mutate(
        offer_prime_epoch2 = mean(pol[[2]][cbind(seq_len(w$n), w$tier0)] &
          w$tier0 == 1L) / mean(w$tier0 == 1L),
        offer_sub_epoch2 = mean(pol[[2]][cbind(seq_len(w$n), w$tier0)] &
          w$tier0 == 3L) / mean(w$tier0 == 3L)
      )
  })) |>
  unnest(res)

write_csv(sens, "outputs/sensitivity.csv")

sens |>
  select(
    late_to_default, lgd, npv, default_rate_per_accepted,
    offer_prime_epoch2, offer_sub_epoch2
  ) |>
  mutate(across(c(npv), round), across(where(is.double), \(x) round(x, 3))) |>
  kable(digits = 3, format.args = list(big.mark = ","))




| late_to_default|  lgd|       npv| default_rate_per_accepted| offer_prime_epoch2| offer_sub_epoch2|
|---------------:|----:|---------:|-------------------------:|------------------:|----------------:|
|            0.30| 0.35| 1,474,671|                     0.026|              0.937|            0.620|
|            0.30| 0.55| 1,368,800|                     0.025|              1.000|            0.544|
|            0.30| 0.75| 1,269,622|                     0.024|              1.000|            0.493|
|            0.45| 0.35| 1,373,778|                     0.038|              1.000|            0.555|
|            0.45| 0.55| 1,225,252|                     0.037|              1.000|            0.481|
|            0.45| 0.75| 1,080,582|                     0.036|              1.000|            0.431|
|            0.60| 0.35| 1,278,733|                     0.049|              1.000|            0.512|
|            0.60| 0.55| 1,084,533|                     0.048|              1.000|       

In [11]:
# 5.2 Borrower Behavior Sensitivity (Churn Lift, Early Repayment, Acceptance)
eval_constrained <- \(p_mod, logit_shift = 0) {
  w <- build_world(customers, p_mod, accept_logit_shift = logit_shift)
  lam <- solve_constrained(w)
  summarise_evaluation(evaluate_policy(solve_policy(w, lam)$offers, w), w)
}

behavior_sens <- bind_rows(
  eval_constrained(p) |> mutate(experiment = "Base (churn_acc=0.01, p_early=0.20)"),
  eval_constrained(list_modify(p, churn_accepted = 0.04)) |>
    mutate(experiment = "No retention lift (churn_acc=0.04)"),
  eval_constrained(list_modify(p, p_early = 0.00)) |>
    mutate(experiment = "Zero early repayment (p_early=0.00)"),
  eval_constrained(list_modify(p, p_early = 0.40)) |>
    mutate(experiment = "High early repayment (p_early=0.40)"),
  eval_constrained(p, logit_shift = -0.30) |>
    mutate(experiment = "Lower uptake (logit -0.30)"),
  eval_constrained(p, logit_shift = 0.30) |>
    mutate(experiment = "Higher uptake (logit +0.30)")
) |>
  select(experiment, npv, exposure_vs_cap, accepted_increases, default_rate_per_accepted)

cat("Borrower Behavior Sensitivity (re-solved under 30% cap):\n")
print(kable(behavior_sens, digits = 3, format.args = list(big.mark = ",")))

# 5.3 Innovative Strategy: Risk-Tiered Variable Step Sizing (Prime 35%, Near-Prime 20%, Sub-Prime 10%)
# Scale unit profit proportionally with step size ($56 / $32 / $16 vs $40 at 25%)
p_tiered <- list_modify(
  p,
  increase_frac = c(0.35, 0.20, 0.10),
  profit_per_increase = 40 * c(0.35, 0.20, 0.10) / 0.25
)
world_tiered <- build_world(customers, p_tiered)
lam_tiered <- solve_constrained(world_tiered)
tiered_eval <- summarise_evaluation(
  evaluate_policy(solve_policy(world_tiered, lam_tiered)$offers, world_tiered),
  world_tiered
) |>
  mutate(strategy = "dp_tiered_step (35%/20%/10%)")

uniform_eval <- results |>
  filter(scenario == "base", strategy == "dp_constrained") |>
  select(any_of(names(tiered_eval))) |>
  mutate(strategy = "dp_uniform_step (25%)")

cat("\nUniform vs. Risk-Tiered Step Sizing (base scenario, 30% cap):\n")
bind_rows(uniform_eval, tiered_eval) |>
  select(strategy, npv, credit_loss, exposure_vs_cap, accepted_increases, default_rate_per_accepted) |>
  kable(digits = 3, format.args = list(big.mark = ","))


Borrower Behavior Sensitivity (re-solved under 30% cap):


|experiment                          |         npv| exposure_vs_cap| accepted_increases| default_rate_per_accepted|
|:-----------------------------------|-----------:|---------------:|------------------:|-------------------------:|
|Base (churn_acc=0.01, p_early=0.20) | 1,084,533.5|               1|          43,133.11|                     0.048|
|No retention lift (churn_acc=0.04)  | 1,063,523.2|               1|          42,731.74|                     0.048|
|Zero early repayment (p_early=0.00) | 1,067,326.7|               1|          43,131.30|                     0.049|
|High early repayment (p_early=0.40) | 1,101,061.6|               1|          43,131.55|                     0.046|
|Lower uptake (logit -0.30)          |   919,331.8|               1|          39,976.13|                     0.050|
|Higher uptake (logit +0.30)         | 1,228,814.1|               1|          45,607.28|                     0.045|



Uniform vs. Risk-Tiered Step Sizing (base scenario, 30% cap):




|strategy                     |       npv| credit_loss| exposure_vs_cap| accepted_increases| default_rate_per_accepted|
|:----------------------------|---------:|-----------:|---------------:|------------------:|-------------------------:|
|dp_uniform_step (25%)        | 1,084,533|   527,067.5|               1|          43,133.11|                     0.048|
|dp_tiered_step (35%/20%/10%) | 1,144,109|   393,756.1|               1|          41,273.19|                     0.047|

## 6. Efficient Frontier & Visual Diagnostics

Modernized `ggplot2 >= 4.0` visualizations using `theme_minimal(ink =, paper =, accent =)`, `element_geom()`, `theme_sub_*()` shortcuts, `labs(dictionary = ...)`, direct value annotations, and `patchwork` composition. Each chart is saved to `figures/` and displayed inline.


In [12]:
# Compute capital efficient frontier across shadow prices lambda (including lambda_star)
frontier <- tibble(
  lambda = sort(unique(c(0, lambda_star, seq(0.02, 1, length.out = 24))))
) |>
  mutate(res = map(lambda, \(l) {
    summarise_evaluation(
      evaluate_policy(solve_policy(world_base, l)$offers, world_base),
      world_base
    )
  })) |>
  unnest(res)

# Summarise epoch-2 offer rates by risk tier and initial loan band
policy_tbl <- tibble(
  tier = factor(tier_labels[world_base$tier0], levels = tier_labels),
  loan = pull(customers, initial_loan),
  offered = policies_base$dp_constrained[[2]][cbind(
    seq_len(world_base$n), world_base$tier0
  )]
) |>
  mutate(loan_band = cut_width(loan, 500, boundary = 0, dig.lab = 4))

policy_summary <- policy_tbl |>
  summarise(
    n = n(),
    mean_loan = mean(loan),
    offer_share = mean(offered),
    .by = c(tier, loan_band)
  ) |>
  arrange(tier, loan_band)

write_csv(policy_summary, "outputs/policy_summary_by_tier_loan.csv")

# Shared ggplot2 4.0+ theme, label dictionary, and palettes
theme_loan <- function() {
  theme_minimal(
    base_size = 11, ink = "#1b2a41", paper = "#fbfaf7", accent = "#c8553d"
  ) +
    theme(geom = element_geom(ink = "#1b2a41", accent = "#c8553d", linewidth = 0.9)) +
    theme_sub_panel(
      grid.minor = element_blank(),
      grid.major = element_line(colour = "#e5e2dc", linewidth = 0.35)
    ) +
    theme_sub_strip(
      text = element_text(face = "bold", size = 10.5),
      background = element_rect(fill = "#f1eee8", colour = NA)
    ) +
    theme_sub_legend(position = "bottom") +
    theme_sub_plot(
      title.position = "plot",
      title = element_text(face = "bold", size = 12),
      subtitle = element_text(colour = "#4a5568", size = 9.5),
      margin = margin(8, 10, 8, 10)
    )
}

labels_dict <- c(
  npv = "Portfolio NPV",
  strategy = "Strategy",
  scenario = "Macro scenario",
  exposure_vs_cap = "Expected new exposure / capital cap",
  lgd = "Loss given default (LGD)",
  late_to_default = "Late-to-default conversion",
  loan = "Initial loan ($)",
  breakeven_pd = "Break-even PD per 60 days",
  tier = "Risk tier",
  loan_band = "Initial loan band ($)",
  offer_share = "Offered at epoch 2"
)

usd <- label_dollar(scale_cut = cut_short_scale())
strategy_levels <- c(
  "no_increase", "offer_all", "prime_only",
  "prime_near", "dp_unconstrained", "dp_constrained"
)
strategy_labels <- c(
  "No Increase", "Offer All", "Prime Only",
  "Prime + Near", "DP Free", "DP Constrained"
)
tier_palette <- c(
  "Prime" = "#2a9d8f",
  "Near-Prime" = "#d97706",
  "Sub-Prime" = "#c8553d"
)


In [13]:
# 6.1 Strategy NPV Comparison (horizontal bars eliminate angled labels)
p_strat <- results |>
  filter(strategy %in% strategy_levels) |>
  mutate(
    strategy = factor(strategy, levels = strategy_levels, labels = strategy_labels),
    scenario = fct_relabel(scenario, str_to_title),
    cap_status = if_else(within_cap, "Within Cap", "Breaches Cap")
  ) |>
  ggplot(aes(npv, fct_rev(strategy), fill = cap_status)) +
  geom_col(width = 0.68) +
  geom_vline(xintercept = 0, linewidth = 0.35, colour = "#1b2a41") +
  geom_text(
    aes(
      label = usd(npv),
      hjust = if_else(npv >= 0, -0.1, 1.1)
    ),
    size = 2.7,
    fontface = "bold",
    colour = "#1b2a41"
  ) +
  facet_wrap(vars(scenario)) +
  scale_x_continuous(labels = usd, expand = expansion(mult = c(0.15, 0.46))) +
  scale_fill_manual(
    values = c("Within Cap" = "#2a9d8f", "Breaches Cap" = "#c8553d"),
    name = NULL
  ) +
  labs(
    title = "Portfolio NPV by strategy and macro scenario",
    subtitle = "DP Constrained maximizes value while respecting the 30% capital cap",
    y = NULL,
    dictionary = labels_dict
  ) +
  theme_loan()

ggsave("figures/r_01_strategy_npv.png", p_strat, width = 10, height = 4.8, dpi = 150)


In [14]:
# 6.2 Capital-Constrained Optimal Policy Heatmap
p_policy <- policy_summary |>
  ggplot(aes(loan_band, tier, fill = offer_share)) +
  geom_tile(colour = "#fbfaf7", linewidth = 0.6) +
  geom_text(
    aes(
      label = percent(offer_share, accuracy = 1),
      colour = if_else(offer_share > 0.55, "#1b2a41", "white")
    ),
    size = 2.8,
    fontface = "bold"
  ) +
  scale_colour_identity() +
  scale_fill_viridis_c(
    option = "mako",
    labels = label_percent(),
    name = "Offered at epoch 2",
    guide = guide_colorbar(barwidth = unit(9, "lines"))
  ) +
  guides(x = guide_axis(angle = 35)) +
  labs(
    title = "Capital-constrained optimal policy: who gets an offer",
    subtitle = "Smaller loans and lower-risk tiers earn more NPV per dollar of capital",
    dictionary = labels_dict
  ) +
  theme_loan()

ggsave("figures/r_02_policy_map.png", p_policy, width = 9.5, height = 4.2, dpi = 150)


In [15]:
# 6.3 Efficient Frontier: NPV vs Capital Utilization
opt_pts <- results |>
  filter(scenario == "base", strategy %in% c("dp_constrained", "dp_unconstrained")) |>
  mutate(label = if_else(
    strategy == "dp_constrained",
    paste0("DP Constrained\n", usd(npv), " (100% cap)"),
    paste0("DP Free\n", usd(npv), " (", percent(exposure_vs_cap, accuracy = 1), " cap)")
  ))

p_frontier <- frontier |>
  ggplot(aes(exposure_vs_cap, npv)) +
  geom_area(alpha = 0.08, fill = "#2a9d8f") +
  geom_line(linewidth = 0.9, colour = "#1b2a41") +
  geom_point(size = 1.8, colour = "#1b2a41") +
  geom_vline(xintercept = 1, linetype = "dashed", colour = "#c8553d", linewidth = 0.7) +
  geom_point(data = opt_pts, aes(colour = strategy), size = 3.2, show.legend = FALSE) +
  ggrepel::geom_label_repel(
    data = opt_pts,
    aes(label = label, colour = strategy),
    size = 2.8,
    fontface = "bold",
    fill = "#fbfaf7",
    box.padding = 0.5,
    show.legend = FALSE
  ) +
  scale_colour_manual(values = c("dp_constrained" = "#2a9d8f", "dp_unconstrained" = "#c8553d")) +
  scale_x_continuous(labels = label_percent()) +
  scale_y_continuous(labels = usd, expand = expansion(mult = c(0.05, 0.14))) +
  labs(
    title = "Efficient frontier: NPV versus capital utilization",
    subtitle = "Dashed line = regulatory cap (30% of initial loan book)",
    dictionary = labels_dict
  ) +
  theme_loan()

ggsave("figures/r_03_frontier.png", p_frontier, width = 7.5, height = 4.8, dpi = 150)


In [16]:
# 6.4 Monte Carlo Validation Histogram
usd_mc <- label_dollar(scale_cut = cut_short_scale(), accuracy = 0.001)

p_mc <- tibble(npv = mc_npv) |>
  ggplot(aes(npv)) +
  geom_histogram(bins = 30, fill = "#1b2a41", colour = "#fbfaf7", alpha = 0.88) +
  geom_vline(xintercept = c(mc_q05, mc_q95), linetype = "dashed", colour = "#4a5568", linewidth = 0.6) +
  geom_vline(xintercept = analytic_npv, colour = "#c8553d", linewidth = 1.1) +
  scale_x_continuous(labels = usd) +
  labs(
    title = "Monte Carlo NPV vs analytic value (1,000 replications)",
    subtitle = sprintf(
      "Solid line = Analytic (%s) | Dashed = 5th-95th pct | MC mean = %s (z = %.2f)",
      usd_mc(analytic_npv), usd_mc(mean(mc_npv)), mc_check$z
    ),
    x = "Portfolio NPV",
    y = "Simulations"
  ) +
  theme_loan()

ggsave("figures/r_04_monte_carlo.png", p_mc, width = 7.5, height = 4.8, dpi = 150)


In [17]:
# 6.5 Sensitivity Heatmap (with base-case highlight and adaptive contrast text)
base_cell <- sens |>
  filter(late_to_default == 0.6, lgd == 0.55) |>
  mutate(
    late_to_default = factor(late_to_default),
    lgd = factor(lgd)
  )

p_sens <- sens |>
  mutate(
    late_to_default = factor(late_to_default),
    lgd = factor(lgd)
  ) |>
  ggplot(aes(late_to_default, lgd, fill = npv)) +
  geom_tile(colour = "#fbfaf7", linewidth = 0.6) +
  geom_tile(data = base_cell, fill = NA, colour = "#c8553d", linewidth = 1.4) +
  geom_text(
    aes(
      label = usd(npv),
      colour = if_else(npv > 1.15e6, "#1b2a41", "white")
    ),
    size = 3.2,
    fontface = "bold"
  ) +
  scale_colour_identity() +
  scale_fill_viridis_c(
    option = "mako",
    labels = usd,
    name = "NPV",
    breaks = c(7e5, 1.0e6, 1.3e6),
    guide = guide_colorbar(barwidth = unit(10, "lines"))
  ) +
  labs(
    title = "Sensitivity of optimal NPV to credit assumptions",
    subtitle = "Outlined cell marks the base-case assumption (late_to_default = 0.60, LGD = 0.55)",
    dictionary = labels_dict
  ) +
  theme_loan()

ggsave("figures/r_05_sensitivity.png", p_sens, width = 7.5, height = 4.5, dpi = 150)


In [18]:
# 6.6 One-Cycle Constrained Break-Even Default Rate vs Loan Size
breakeven <- expand_grid(loan = seq(500, 5000, 100)) |>
  mutate(
    ead_i = p$increase_frac * loan,
    breakeven_pd = (p$profit_per_increase - lambda_star * ead_i) /
      (world_base$disc * p$lgd * ead_i)
  )

tier_lines <- tier_table |>
  mutate(tier = factor(tier_labels[tier], levels = tier_labels))

p_be <- breakeven |>
  ggplot(aes(loan, breakeven_pd)) +
  geom_line(linewidth = 1.1, colour = "#1b2a41") +
  geom_hline(
    data = tier_lines,
    aes(yintercept = pd_cycle, colour = tier),
    linetype = "dashed",
    linewidth = 0.8
  ) +
  geom_text(
    data = tier_lines,
    aes(
      x = 550,
      y = pd_cycle,
      label = paste0(tier, " (", percent(pd_cycle, accuracy = 0.1), ")"),
      colour = tier
    ),
    hjust = 0,
    vjust = -0.5,
    size = 2.9,
    fontface = "bold",
    show.legend = FALSE
  ) +
  scale_colour_manual(values = tier_palette, name = "Risk tier") +
  scale_x_continuous(labels = label_dollar()) +
  scale_y_continuous(labels = label_percent()) +
  coord_cartesian(ylim = c(0, 0.60)) +
  labs(
    title = "Terminal-epoch (myopic) break-even PD vs loan size",
    subtitle = "Single-cycle hurdle at lambda*; multi-period DP also weighs retention lift & tier-upgrade value",
    dictionary = labels_dict
  ) +
  theme_loan()

ggsave("figures/r_06_breakeven.png", p_be, width = 7.5, height = 4.5, dpi = 150)


In [19]:
# 6.7 Combined 6-Panel Executive Dashboard via patchwork
options(ark.plot.width = 16, ark.plot.height = 14)

p_dashboard <- (p_strat + p_policy) / (p_frontier + p_mc) / (p_sens + p_be) +
  plot_annotation(
    title = "Loan Limit Increase Optimization: Executive Diagnostic Dashboard",
    subtitle = "30,000-customer portfolio | 60-day MDP (K = 6 epochs/yr) | 30% capital cap",
    caption = "Source: data/loan_limit_increases.csv & 1,000-rep Monte Carlo validation",
    theme = theme_loan() +
      theme_sub_plot(
        title = element_text(face = "bold", size = 15),
        subtitle = element_text(colour = "#4a5568", size = 11),
        caption = element_text(colour = "#718096", size = 9)
      )
  )

ggsave("figures/r_dashboard.png", p_dashboard, width = 16, height = 14, dpi = 150)
p_dashboard


## 7. Headline Summary & Operationalization

Comparing the capital-constrained DP optimum against the naive `offer_all` and heuristic `prime_near` baselines in the `base` macro scenario, followed by production deployment recommendations:

### Operationalizing in a Real Lending Environment
1. **Rolling-Horizon Execution**: Re-solve the 6-epoch DP after every 60-day cycle so epoch 6 never hits an artificial $V_7 = 0$ terminal boundary.
2. **Online Shadow-Price ($\lambda^*$) Recalibration**: Update $\lambda^*$ monthly as realized macro indicators (inflation, unemployment, policy rate) and portfolio capital utilization shift.
3. **Dynamic Demand & Continuous Risk Scoring**: Replace the static Bernoulli acceptance rate with a two-component responder mixture (Bayesian posterior updating after each accept/decline) and replace discrete tier cutoffs with continuous PD scoring to eliminate the $3.9	imes$ cliff at `on_time_pct = 95%`.
4. **Tiered Step Sizing & Guardrail Monitoring**: Deploy smaller limit steps for higher-risk tiers (`10%`–`20%`) and maintain a 5%–10% randomized holdout cell to continuously validate uplift, churn, and realized LGD.


In [20]:
head_tbl <- results |>
  filter(
    scenario == "base",
    strategy %in% c("offer_all", "prime_near", "dp_constrained")
  ) |>
  select(strategy, npv, exposure_vs_cap, within_cap)

kable(head_tbl, digits = 3, format.args = list(big.mark = ","))




|strategy       |       npv| exposure_vs_cap|within_cap |
|:--------------|---------:|---------------:|:----------|
|offer_all      | 1,043,611|           1.621|FALSE      |
|prime_near     | 1,011,908|           1.224|FALSE      |
|dp_constrained | 1,084,533|           1.000|TRUE       |